# 如何使用新的 seed 参数让补全输出可复现

**太长不看**：开发者现在可以在 Chat Completion 请求中指定 `seed` 参数，以获得（基本）一致的输出。为了帮助你跟踪这些变化，我们提供了 `system_fingerprint` 字段。如果该值发生变化，你可能会因为我们对系统所做的更改而看到不同的输出。请注意，此功能处于 beta 阶段，目前仅支持 `gpt-4-1106-preview` 和 `gpt-3.5-turbo-1106`。

### 背景

可复现性一直是用户社区在使用我们的 API 时提出的一项重要需求。例如，一旦能够获得可复现的数值结果，用户就可以解锁相当多对数值变化敏感的用例。

#### 实现一致输出的模型级功能

Chat Completions 和 Completions API 默认是非确定性的（这意味着模型输出可能因请求而异），但现在提供了一些模型级控制，可在一定程度上实现确定性输出。

这可以带来一致的补全结果，让基于这些 API 构建的任何应用都能完全掌控模型行为；对于复现结果和测试也非常有用，让你确切知道会得到什么，从而更加安心。

#### 实现一致的输出

要在多次 API 调用中获得_基本_确定的输出：

- 将 `seed` 参数设置为你选择的任意整数，但在各次请求中使用相同的值。例如 `12345`。
- 在各次请求中将所有其他参数（prompt、temperature、top_p 等）设置为相同的值。
- 在响应中检查 `system_fingerprint` 字段。系统指纹是 OpenAI 服务器生成补全时所用的模型权重、基础设施和其他配置选项当前组合的标识符。每当你更改请求参数，或 OpenAI 更新为模型提供服务的基础设施的数值配置（一年可能发生几次）时，它都会改变。

如果各次请求的 `seed`、请求参数和 `system_fingerprint` 都一致，那么模型输出基本上会是相同的。由于我们的模型固有的非确定性，即使请求参数和 `system_fingerprint` 一致，响应仍有小概率不同。


### 实现一致输出的模型级控制——`seed` 和 `system_fingerprint`

##### `seed`

如果指定了该参数，我们的系统会尽最大努力进行确定性采样，使得使用相同 seed 和参数的重复请求返回相同的结果。确定性无法保证，你应参考响应参数 `system_fingerprint` 来监控后端的变化。

##### `system_fingerprint`

该指纹代表模型运行时所用的后端配置。它可以与请求参数 seed 结合使用，用来了解后端何时发生了可能影响确定性的变化。它是用户是否应当预期“几乎总是相同的结果”的指示器。


## 示例：使用固定 seed 生成一段简短摘录

在本例中，我们将演示如何使用固定的 seed 生成一段简短摘录。当你需要为测试、调试生成一致的结果，或者应用本身需要一致的输出时，这尤其有用。

### Python SDK

> **注意**
> 请切换到最新版本的 SDK（撰写本文时为 `1.3.3`）。

In [ ]:
!pip install --upgrade openai # Switch to the latest version of OpenAI (1.3.3 at time of writing)

In [12]:
import openai
import asyncio
from IPython.display import display, HTML

from utils.embeddings_utils import (
    get_embedding,
    distances_from_embeddings
)

GPT_MODEL = "gpt-3.5-turbo-1106"

In [13]:
async def get_chat_response(
    system_message: str, user_request: str, seed: int = None, temperature: float = 0.7
):
    try:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_request},
        ]

        response = openai.chat.completions.create(
            model=GPT_MODEL,
            messages=messages,
            seed=seed,
            max_tokens=200,
            temperature=temperature,
        )

        response_content = response.choices[0].message.content
        system_fingerprint = response.system_fingerprint
        prompt_tokens = response.usage.prompt_tokens
        completion_tokens = response.usage.total_tokens - response.usage.prompt_tokens

        table = f"""
        <table>
        <tr><th>Response</th><td>{response_content}</td></tr>
        <tr><th>System Fingerprint</th><td>{system_fingerprint}</td></tr>
        <tr><th>Number of prompt tokens</th><td>{prompt_tokens}</td></tr>
        <tr><th>Number of completion tokens</th><td>{completion_tokens}</td></tr>
        </table>
        """
        display(HTML(table))

        return response_content
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def calculate_average_distance(responses):
    """
    This function calculates the average distance between the embeddings of the responses.
    The distance between embeddings is a measure of how similar the responses are.
    """
    # Calculate embeddings for each response
    response_embeddings = [get_embedding(response) for response in responses]

    # Compute distances between the first response and the rest
    distances = distances_from_embeddings(response_embeddings[0], response_embeddings[1:])

    # Calculate the average distance
    average_distance = sum(distances) / len(distances)

    # Return the average distance
    return average_distance

首先，我们在不使用 `seed` 参数的情况下，尝试生成几个关于“火星之旅”的简短摘录的不同版本。这是默认行为：

In [14]:
topic = "a journey to Mars"
system_message = "You are a helpful assistant."
user_request = f"Generate a short excerpt of news about {topic}."

responses = []


async def get_response(i):
    print(f'Output {i + 1}\n{"-" * 10}')
    response = await get_chat_response(
        system_message=system_message, user_request=user_request
    )
    return response


responses = await asyncio.gather(*[get_response(i) for i in range(5)])
average_distance = calculate_average_distance(responses)
print(f"The average similarity between responses is: {average_distance}")

Output 1
----------


Response,"""NASA's Mars mission reaches critical stage as spacecraft successfully enters orbit around the red planet. The historic journey, which began over a year ago, has captured the world's attention as scientists and astronauts prepare to land on Mars for the first time. The mission is expected to provide valuable insights into the planet's geology, atmosphere, and potential for sustaining human life in the future."""
System Fingerprint,fp_772e8125bb
Number of prompt tokens,29
Number of completion tokens,76


Output 2
----------


Response,"""NASA's Perseverance rover successfully landed on Mars, marking a major milestone in the mission to explore the red planet. The rover is equipped with advanced scientific instruments to search for signs of ancient microbial life and collect samples of rock and soil for future return to Earth. This historic achievement paves the way for further exploration and potential human missions to Mars in the near future."""
System Fingerprint,fp_772e8125bb
Number of prompt tokens,29
Number of completion tokens,76


Output 3
----------


...

Output 4
----------


Response,"""NASA's latest Mars mission exceeds expectations as the Perseverance rover uncovers tantalizing clues about the Red Planet's past. Scientists are thrilled by the discovery of ancient riverbeds and sedimentary rocks, raising hopes of finding signs of past life on Mars. With this exciting progress, the dream of sending humans to Mars feels closer than ever before."""
System Fingerprint,fp_772e8125bb
Number of prompt tokens,29
Number of completion tokens,72


Output 5
----------


Response,"""NASA's Perseverance Rover Successfully Lands on Mars, Begins Exploration Mission In a historic moment for space exploration, NASA's Perseverance rover has successfully landed on the surface of Mars. After a seven-month journey, the rover touched down in the Jezero Crater, a location scientists believe may have once held a lake and could potentially contain signs of ancient microbial life. The rover's primary mission is to search for evidence of past life on Mars and collect rock and soil samples for future return to Earth. Equipped with advanced scientific instruments, including cameras, spectrometers, and a drill, Perseverance will begin its exploration of the Martian surface, providing valuable data and insights into the planet's geology and potential habitability. This successful landing marks a significant milestone in humanity's quest to understand the red planet and paves the way for future manned missions to Mars. NASA's Perseverance rover is poised to unravel the mysteries of Mars and unlock new possibilities"
System Fingerprint,fp_772e8125bb
Number of prompt tokens,29
Number of completion tokens,200


The average similarity between responses is: 0.1136714512418833


现在，我们用固定的 `seed` 值 123 和 `temperature` 值 0 运行同样的代码，并比较响应和 `system_fingerprint`。

In [15]:
SEED = 123
responses = []


async def get_response(i):
    print(f'Output {i + 1}\n{"-" * 10}')
    response = await get_chat_response(
        system_message=system_message,
        seed=SEED,
        temperature=0,
        user_request=user_request,
    )
    return response


responses = await asyncio.gather(*[get_response(i) for i in range(5)])

average_distance = calculate_average_distance(responses)
print(f"The average distance between responses is: {average_distance}")

Output 1
----------


Response,"""NASA's Perseverance Rover Successfully Lands on Mars In a historic achievement, NASA's Perseverance rover has successfully landed on the surface of Mars, marking a major milestone in the exploration of the red planet. The rover, which traveled over 293 million miles from Earth, is equipped with state-of-the-art instruments designed to search for signs of ancient microbial life and collect rock and soil samples for future return to Earth. This mission represents a significant step forward in our understanding of Mars and the potential for human exploration of the planet in the future."""
System Fingerprint,fp_772e8125bb
Number of prompt tokens,29
Number of completion tokens,113


Output 2
----------


Response,"""NASA's Perseverance rover successfully lands on Mars, marking a historic milestone in space exploration. The rover is equipped with advanced scientific instruments to search for signs of ancient microbial life and collect samples for future return to Earth. This mission paves the way for future human exploration of the red planet, as scientists and engineers continue to push the boundaries of space travel and expand our understanding of the universe."""
System Fingerprint,fp_772e8125bb
Number of prompt tokens,29
Number of completion tokens,81


Output 3
----------


...

Output 4
----------


Response,"""NASA's Perseverance rover successfully lands on Mars, marking a historic milestone in space exploration. The rover is equipped with advanced scientific instruments to search for signs of ancient microbial life and collect samples for future return to Earth. This mission paves the way for future human exploration of the red planet, as scientists and engineers continue to push the boundaries of space travel and expand our understanding of the universe."""
System Fingerprint,fp_772e8125bb
Number of prompt tokens,29
Number of completion tokens,81


Output 5
----------


Response,"""NASA's Perseverance rover successfully lands on Mars, marking a historic milestone in space exploration. The rover is equipped with advanced scientific instruments to search for signs of ancient microbial life and collect samples for future return to Earth. This mission paves the way for future human exploration of the red planet, as scientists and engineers continue to push the boundaries of space travel."""
System Fingerprint,fp_772e8125bb
Number of prompt tokens,29
Number of completion tokens,74


The average distance between responses is: 0.0449054397632461


可以看到，`seed` 参数让我们能够生成一致得多的结果。

## 总结

我们演示了如何使用固定的整数 `seed` 从模型生成一致的输出。这在可复现性很重要的场景中尤其有用。不过需要注意，虽然 `seed` 能确保一致性，但并不能保证输出的质量。注意，当你想使用可复现的输出时，需要在各次 Chat Completions 调用中将 `seed` 设置为同一个整数。你还应保持其他参数（如 `temperature`、`max_tokens` 等）一致。可复现输出的进一步扩展用法是：在对不同的提示或模型进行基准测试/评估时使用一致的 `seed`，以确保每个版本都在相同条件下接受评估，使比较公平、结果可靠。